# Prédiction

## Occupation en fonction de l'heure

In [1]:
from ipywidgets import HBox, VBox, Box, fixed, interactive_output
from plotly.graph_objects import FigureWidget
from IPython.display import display

import pandas as pd
import plotly.express as px

In [7]:
df_occupation = pd.read_csv('./Followchon/data/df_occupation.csv')

df_occupation

,date,datehour,zone,hour,occupation,class,T,zone_id,class_index
0,2024-07-21,2024-07-21 15:00:00,Clapier,15,0.018056,Noisette,26.6,1.0,1
1,2024-07-21,2024-07-21 16:00:00,Clapier,16,0.003333,Noisette,25.5,1.0,1
2,2024-07-21,2024-07-21 17:00:00,Clapier,17,0.001111,Noisette,25.1,1.0,1
3,2024-07-21,2024-07-21 18:00:00,Clapier,18,0.005278,Noisette,25.1,1.0,1
4,2024-07-21,2024-07-21 19:00:00,Clapier,19,0.000278,Noisette,24.7,1.0,1
...,...,...,...,...,...,...,...,...,...
7149,2024-11-03,2024-11-03 16:00:00,Foin,16,0.099722,Stitch,13.2,16.0,2
7150,2024-11-03,2024-11-03 14:00:00,Tunnel maison,14,0.000000,Stitch,14.3,17.0,2
7151,2024-11-03,2024-11-03 14:00:00,Tunnel,14,0.000000,Stitch,14.3,12.0,2
7152,2024-11-03,2024-11-03 15:00:00,Tunnel,15,0.000000,Stitch,14.4,12.0,2


- Parle-t-on de problème supervisée ou non supervisée ?
  - On est face à un problème supervisé car nous avons des données à disposition pour l'entrainement
- Les données sont-elles stucturées ou non structurées ?
    - Les données sont structurées avec en ligne les dates et en colonnes les features (suivant le modèle relationnel)
- Est-ce un problème de régression ou de classification ? Aucun des deux ?
    - Si on cherche à déterminer l'occupation en fonction de l'heure sur une zone déterminé, c'est un problème de régression
    - Si on cherche à déterminer la zone occupé en fonction de l'heure, c'est un problème de classification
- Quelles sont les features et quelle est la target ?
    - Target :
        - Soit l'occupation
        - Soit la zone
    - Features :
        - date
        - heure
        - température
        - class

- suppression des colonnes inutiles (customer_id)
- suppressions des valeurs absurdes (valeurs négatives de average_price)
- dummification des valeurs textuelles avec labelencoder pour la target (client_type) et onehotencoder pour les features (country, gender)
- normalisation des features numériques (average_basket, average_price, visit_number, age)
- concaténation des résultats dans une seule variable X

In [3]:
hour_begin = 9
hour_end = 19

df = df_occupation.drop("datehour", axis=1)
df = df[(df['zone'] == 'Cachette')]
df = df[df['hour'].between(hour_begin, hour_end)]

indices_to_drop = df[
    (df['occupation'] < 0.34) | (df['occupation'] > 0.42)
].index

df_filtered = df.drop(indices_to_drop)

In [4]:
def group_by_hour(df):
    return df.loc[:, ['hour', 'class', 'class_index', 'occupation']]\
        .groupby(['hour', 'class', 'class_index'])\
        .mean('occupation')\
        .reset_index()

def show_scatter(df, title, range_y):
    fig = px.scatter(
            df, 
            title=title,
            x='hour', 
            y='occupation', 
            trendline='lowess',
            trendline_options=dict(frac=0.9),
            range_y=range_y
        )
    
    fig.update_layout(width=700, height=700)
    
    return FigureWidget(fig)

display(
    Box([
        show_scatter(df, 'Raw', [0, 1]),
        show_scatter(group_by_hour(group_by_hour(df)), 'Raw group by hour', [0.34, 0.42]),
    ]),
    Box([
        show_scatter(df_filtered, 'Filtered', [0, 1]),
        show_scatter(group_by_hour(group_by_hour(df_filtered)), 'Filtered group by hour', [0.34, 0.42])
    ])
)

Box(children=(FigureWidget({
    'data': [{'hovertemplate': 'hour=%{x}<br>occupation=%{y}<extra></extra>',
   …

Box(children=(FigureWidget({
    'data': [{'hovertemplate': 'hour=%{x}<br>occupation=%{y}<extra></extra>',
   …